## Projet Optimisation Numérique v0

Dans ce projet, nous nous proposons de développer une plateforme complète d’optimisation automatique appliquée à un problème bidimensionnel de conduction thermique stationnaire résolu avec FreeFEM++. L’objectif principal est de coupler un solveur éléments finis à une méthode d’optimisation externe en Python afin de rechercher automatiquement les paramètres thermiques optimaux maximisant la performance du système.

Le problème étudié repose sur le benchmark HEAT-COND, dans lequel la conductivité thermique de plusieurs couches de matériaux ainsi que le nombre de Biot constituent les variables de conception. Pour chaque configuration de paramètres, le solveur FreeFEM++ résout l’équation de conduction thermique et calcule une fonction objectif correspondant à la température moyenne sur les surfaces des ailettes.

La plateforme développée repose sur une architecture modulaire permettant d’automatiser l’ensemble du workflow numérique : génération des paramètres, résolution du problème PDE, évaluation de la fonction objectif, optimisation et sauvegarde des résultats. L’optimisation est pilotée depuis Python à l’aide des bibliothèques scientifiques standards, tandis que FreeFEM++ assure la résolution numérique du problème physique.

Le pipeline complet du système est le suivant :

1. Python génère un nouveau vecteur de paramètres : x = (k1, k2, k3, k4, k5, Bi)

2. La fonction write_params(x) écrit ces paramètres dans : params.txt

3. La fonction run_freefem() lance automatiquement : FreeFem++ heat_solver.edp

4. Le script FreeFEM ouvre et lit : params.txt

5. FreeFEM construit le domaine thermique et résout le problème PDE
   de conduction de chaleur avec les paramètres reçus.

6. FreeFEM calcule la fonction objectif : J = température moyenne / performance thermique

7. FreeFEM écrit cette valeur dans : objective.txt

8. Python utilise read_objective() pour lire la valeur de J.

9. L’algorithme d’optimisation analyse cette performance :
   
   - si J est meilleur → conservation du design
   - sinon → rejet du design

10. L’optimiseur génère un nouveau vecteur x et le cycle recommence.

11. Le processus continue jusqu’au critère d’arrêt :
    
    - nombre maximal d’itérations
    - convergence
    - tolérance atteinte

**1. Chargement des bibliothèques**

In [1]:
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
import time

**2. Définitions des chemins**

In [2]:
# Définir les chemins
from dotenv import load_dotenv
load_dotenv()  # Charger les variables d'environnement depuis le fichier .env

FREEFEM_EXEC = os.getenv("FREEFEM_PATH")
FREEFEM_SCRIPT = "heat_solver.edp"

if FREEFEM_EXEC is None:
    raise ValueError("Err")

print(FREEFEM_EXEC)

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)  # Crée le dossier pour stocker les résultats

C:\\Program Files\\FreeFem++\\FreeFem++.exe


**3. Ecriture des paramètres d'optimisation dans params.txt**

On définit la fonction write_params() qui sert à écrire les paramètres d’optimisation dans le fichier params.txt, afin que le script FreeFEM++ puisse les lire ensuite.

In [3]:
def write_params(x):
    with open("params.txt", "w") as f:
        for value in x:
            f.write(f"{value}\n")

**4. Lancement automatique du script FreeFEM++**

On définit la fonction run_freefem() qui sert à lancer automatiquement le script FreeFEM++ depuis Python.

In [4]:
def run_freefem(command, arg):
    result = subprocess.run(
        [FREEFEM_EXEC, FREEFEM_SCRIPT, command, arg],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("FreeFEM execution failed")

**5. Lecture de la fonction objectif calculée**

On définit la fonction read_objective() qui sert à lire la valeur de la fonction objectif J calculée par FreeFEM++.

In [5]:
def read_objective():
    with open("objective.txt", "r") as f:
        J = float(f.readline().strip())
    return J

**6. Sauvegarde du meilleur design trouvé lors de la phase d'optimisation**

Cette fonction sert à sauvegarder automatiquement le meilleur design trouvé pendant l’optimisation.

Autrement dit, dès qu’un meilleur J est trouvé, on écrit les paramètres optimaux dans un fichier texte.

In [6]:
def save_best_design():
    global best_J, best_x

    filepath = os.path.join(RESULTS_DIR, "best_design.txt")

    with open(filepath, "w") as f:
        f.write("===== BEST DESIGN =====\n\n")
        f.write(f"k1 = {best_x[0]}\n")
        f.write(f"k2 = {best_x[1]}\n")
        f.write(f"k3 = {best_x[2]}\n")
        f.write(f"k4 = {best_x[3]}\n")
        f.write(f"k5 = {best_x[4]}\n")
        f.write(f"Bi = {best_x[5]}\n\n")
        f.write(f"Best objective J = {best_J}\n")

**7. Définition de la fonction d'évaluation**

On définit la fonction d’évaluation pour une optimisation qui envoie des paramètres vers une simulation FreeFEM++, récupère la valeur d’objectif J, et mesure le temps d’exécution.
Il enregistre chaque essai (paramètres, résultat, temps) dans un historique et garde en mémoire la meilleure solution trouvée.
Enfin, il retourne −J pour permettre une optimisation par minimisation avec SciPy.

In [7]:
history = []  # Liste globale pour stocker l'historique des évaluations
iteration_counter = 0  # Compteur d'itérations
best_J = -np.inf  # Meilleur objectif
best_x = None  # Meilleurs paramètres
nombre_iterations_max = 10  # Nombre maximum d'itérations

def evaluate(x):
    global iteration_counter, best_J, best_x

    if iteration_counter > nombre_iterations_max:
        # Sauvegarder les meilleures valeurs dans un fichier final
        filepath = os.path.join("./", "params_final.txt")
        with open(filepath, "w") as f:
            f.write(f"{best_x[0]}\n")
            f.write(f"{best_x[1]}\n")
            f.write(f"{best_x[2]}\n")
            f.write(f"{best_x[3]}\n")
            f.write(f"{best_x[4]}\n")
            f.write(f"{best_x[5]}\n\n")
        return float(np.inf)  # Arrêter l'évaluation

    start_time = time.time()

    # Sauvegarder les paramètres
    write_params(x)

    # Lancer FreeFEM++
    run_freefem("-doplot", "0")  # Ne pas afficher les graphiques pendant l'optimisation

    # Lire l'objectif
    J = read_objective()

    elapsed = time.time() - start_time

    # Stocker dans l'historique
    row = {
        "iteration": iteration_counter,
        "k1": x[0],
        "k2": x[1],
        "k3": x[2],
        "k4": x[3],
        "k5": x[4],
        "Bi": x[5],
        "J": J,
        "time": elapsed
    }
    history.append(row)

    # Suivre le meilleur résultat
    if J > best_J:
        best_J = J
        best_x = np.copy(x)
        save_best_design()  # Fonction pour sauvegarder le meilleur design

        # Affichage console
    print("="*60)
    print(f"Iteration : {iteration_counter}")
    print(f"k1 = {x[0]:.6f}")
    print(f"k2 = {x[1]:.6f}")
    print(f"k3 = {x[2]:.6f}")
    print(f"k4 = {x[3]:.6f}")
    print(f"k5 = {x[4]:.6f}")
    print(f"Bi = {x[5]:.6f}")
    print(f"Objective J = {J:.10f}")
    print(f"Evaluation time = {elapsed:.3f} s")
    print(f"Best J so far = {best_J:.10f}")
    print("="*60)

    iteration_counter += 1

    # SciPy minimise, donc on retourne l'opposé de J
    return -J

**8. Sauvegarde de l'historique**

In [8]:
def save_history():
    df = pd.DataFrame(history)
    filepath = os.path.join(RESULTS_DIR, "optimization_history.csv")
    df.to_csv(filepath, index=False)

**9. Visualisation**

In [9]:
def plot_convergence():
    df = pd.DataFrame(history)

    best_values = []
    current_best = -np.inf

    for value in df["J"]:
        current_best = max(current_best, value)
        best_values.append(current_best)

    plt.figure(figsize=(10, 6))
    plt.plot(best_values)
    plt.xlabel("Évaluation")
    plt.ylabel("Meilleur objectif J")
    plt.title("Convergence de l'optimisation")
    plt.grid(True)

    filepath = os.path.join(RESULTS_DIR, "convergence.png")
    plt.savefig(filepath, dpi=300)
    plt.show()
    plt.close()

In [10]:
def save_summary(result, total_time):
    filepath = os.path.join(RESULTS_DIR, "summary.txt")

    with open(filepath, "w") as f:
        f.write("==================================================\n")
        f.write("RÉSUMÉ DE L'OPTIMISATION HEAT-COND\n")
        f.write("==================================================\n\n")
        f.write("Meilleurs paramètres :\n\n")
        f.write(f"k1 = {result.x[0]}\n")
        f.write(f"k2 = {result.x[1]}\n")
        f.write(f"k3 = {result.x[2]}\n")
        f.write(f"k4 = {result.x[3]}\n")
        f.write(f"k5 = {result.x[4]}\n")
        f.write(f"Bi = {result.x[5]}\n\n")
        f.write(f"Meilleur objectif J = {-result.fun}\n\n")
        f.write(f"Nombre d'évaluations = {len(history)}\n")
        f.write(f"Temps total d'optimisation = {total_time:.3f} secondes\n")

In [11]:
def evaluate_initial_design():
    x0 = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    write_params(x0)
    run_freefem("-doplot", "0")
    J0 = read_objective()

    print("\n")
    print("CONCEPTION INITIALE")
    print("======================")
    print(f"J initial = {J0}")
    print("\n")

**10. Programme principal**

In [12]:
# ============================================================
# MAIN PROGRAM
# ============================================================

if __name__ == "__main__":

    print("\n")

    print("===================================================")

    print("HEAT-COND OPTIMIZATION PLATFORM")

    print("===================================================")

    print("\n")

    # --------------------------------------------------------
    # INITIAL DESIGN
    # --------------------------------------------------------

    evaluate_initial_design()

    # --------------------------------------------------------
    # OPTIMIZATION BOUNDS
    # --------------------------------------------------------

    bounds = [
        (0.1, 1.0),   # k1
        (0.1, 1.0),   # k2
        (0.1, 1.0),   # k3
        (0.1, 1.0),   # k4
        (0.1, 1.0),   # k5
        (0.01, 1.0)   # Bi
    ]

    # --------------------------------------------------------
    # START OPTIMIZATION
    # --------------------------------------------------------

    start_total = time.time()

    result = differential_evolution(
        evaluate,
        bounds,
        strategy='best1bin',
        maxiter=10,
        popsize=5,
        tol=1e-4,
        mutation=(0.5, 1.0),
        recombination=0.7,
        polish=True,
        disp=True
    )

    run_freefem("-doplot", "1")  # Afficher les graphiques pour la solution finale

    total_time = time.time() - start_total

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    save_history()

    plot_convergence()

    save_summary(result, total_time)

    # --------------------------------------------------------
    # FINAL OUTPUT
    # --------------------------------------------------------

    print("\n")

    print("===================================================")

    print("OPTIMIZATION FINISHED")

    print("===================================================")

    print("\n")

    print("BEST PARAMETERS FOUND:\n")

    print(f"k1 = {result.x[0]:.8f}")
    print(f"k2 = {result.x[1]:.8f}")
    print(f"k3 = {result.x[2]:.8f}")
    print(f"k4 = {result.x[3]:.8f}")
    print(f"k5 = {result.x[4]:.8f}")
    print(f"Bi = {result.x[5]:.8f}")

    print("\n")

    print(f"Best objective J = {-result.fun:.10f}")

    print(f"Total evaluations = {len(history)}")

    print(f"Total optimization time = {total_time:.3f} seconds")

    print("\n")

    print("Results saved in:")

    print(RESULTS_DIR)

    print("\n")



HEAT-COND OPTIMIZATION PLATFORM


